# Lectura de datos e Inspección inicial

Definir SparkContext y SQLContext


In [1]:
from pyspark import SparkContext
import pyspark
from pyspark.sql import SQLContext
sc = SparkContext()
from pyspark.sql import SQLContext
sqlContext=SQLContext(sc)

ModuleNotFoundError: No module named 'pyspark'

In [2]:
pip install pyspark

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.

     ---------------------------------------- 0.0/434.1 MB ? eta -:--:--
     ---------------------------------------- 0.0/434.1 MB ? eta -:--:--
     -------------------------------------- 0.0/434.1 MB 991.0 kB/s eta 0:07:19
     ---------------------------------------- 0.1/434.1 MB 1.1 MB/s eta 0:06:35
     ---------------------------------------- 0.2/434.1 MB 1.5 MB/s eta 0:04:49
     ---------------------------------------- 0.2/434.1 MB 1.2 MB/s eta 0:06:08
     ---------------------------------------- 0.3/434.1 MB 1.3 MB/s eta 0:05:42
     ---------------------------------------- 0.4/434.1 MB 1.5 MB/s eta 0:04:44
     ---------------------------------------- 0.6/434.1 MB 1.7 MB/s eta 0:04:23
     ---------------------------------------- 0.6/434.1 MB 1.8 MB/s eta 0:04:06
     ---------------------------------------- 0.8/434.1 MB 1.8 MB/s et

In [ ]:
sc.version

'3.5.1'

Lectura del fichero de trabajo

In [3]:
from google.colab import drive
drive.mount('/content/drive')

base_path = '/content/drive/MyDrive/BLOQUE_I.A/Datasets/'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [64]:

bd = sqlContext.read.csv(base_path + "T_ONTIME_REPORTING.csv", header=True, inferSchema=True).repartition(8)


In [65]:
print(bd.rdd.getNumPartitions())


8


In [66]:
type (bd)

pyspark.sql.dataframe.DataFrame

In [67]:
# Look at the first partition contents
bd.rdd.mapPartitionsWithIndex(
    lambda idx, it: [(idx, list(it)[:3])]  # show 3 rows from each partition
).collect()


[(0,
  [Row(YEAR=2024, MONTH=1, DAY_OF_MONTH=11, DAY_OF_WEEK=4, TAIL_NUM='N368DN', ORIGIN_AIRPORT_ID=14027, ORIGIN_AIRPORT_SEQ_ID=1402702, ORIGIN_CITY_MARKET_ID=34027, ORIGIN='PBI', DEST_AIRPORT_ID=12953, DEST_AIRPORT_SEQ_ID=1295304, DEST_CITY_MARKET_ID=31703, DEST='LGA', CRS_DEP_TIME=1955, DEP_DELAY=-10.0, ARR_DELAY=-30.0, CANCELLED=0.0, DIVERTED=0.0, DISTANCE=1035.0, CARRIER_DELAY=None, WEATHER_DELAY=None, NAS_DELAY=None, SECURITY_DELAY=None, LATE_AIRCRAFT_DELAY=None),
   Row(YEAR=2024, MONTH=1, DAY_OF_MONTH=5, DAY_OF_WEEK=5, TAIL_NUM='N8514F', ORIGIN_AIRPORT_ID=10693, ORIGIN_AIRPORT_SEQ_ID=1069302, ORIGIN_CITY_MARKET_ID=30693, ORIGIN='BNA', DEST_AIRPORT_ID=13931, DEST_AIRPORT_SEQ_ID=1393102, DEST_CITY_MARKET_ID=33667, DEST='ORF', CRS_DEP_TIME=1155, DEP_DELAY=0.0, ARR_DELAY=-1.0, CANCELLED=0.0, DIVERTED=0.0, DISTANCE=585.0, CARRIER_DELAY=None, WEATHER_DELAY=None, NAS_DELAY=None, SECURITY_DELAY=None, LATE_AIRCRAFT_DELAY=None),
   Row(YEAR=2024, MONTH=1, DAY_OF_MONTH=5, DAY_OF_WEEK=5, 

## Inspección inicial

Inspección de las variables

In [68]:
bd.dtypes

[('YEAR', 'int'),
 ('MONTH', 'int'),
 ('DAY_OF_MONTH', 'int'),
 ('DAY_OF_WEEK', 'int'),
 ('TAIL_NUM', 'string'),
 ('ORIGIN_AIRPORT_ID', 'int'),
 ('ORIGIN_AIRPORT_SEQ_ID', 'int'),
 ('ORIGIN_CITY_MARKET_ID', 'int'),
 ('ORIGIN', 'string'),
 ('DEST_AIRPORT_ID', 'int'),
 ('DEST_AIRPORT_SEQ_ID', 'int'),
 ('DEST_CITY_MARKET_ID', 'int'),
 ('DEST', 'string'),
 ('CRS_DEP_TIME', 'int'),
 ('DEP_DELAY', 'double'),
 ('ARR_DELAY', 'double'),
 ('CANCELLED', 'double'),
 ('DIVERTED', 'double'),
 ('DISTANCE', 'double'),
 ('CARRIER_DELAY', 'double'),
 ('WEATHER_DELAY', 'double'),
 ('NAS_DELAY', 'double'),
 ('SECURITY_DELAY', 'double'),
 ('LATE_AIRCRAFT_DELAY', 'double')]

Selección de Variables de interés

In [69]:
bd = bd.select(
    'ORIGIN_AIRPORT_ID','YEAR', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'CRS_DEP_TIME',
    'TAIL_NUM', 'ARR_DELAY', 'DEP_DELAY',
    'ORIGIN', 'DEST', 'DISTANCE', 'CANCELLED', 'DIVERTED',
    'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY',
    'LATE_AIRCRAFT_DELAY'
)

# 1) Cuántos registros existen en el dataset.

In [70]:
bd.count()

547271

# 2) Eliminar Diverted

In [71]:
bd = bd.filter(bd.DIVERTED == 0)
print('after removing Diverted', bd.count())

after removing Diverted 545759


# 3) Crear una columna llamada Tarde que abarque los valores DepTime entre las 16 y  las 21 horas ¿Qué porcentaje de los vuelos del horario tarde fueron cancelados?

In [72]:
bd = bd.withColumn('Tarde',(bd.CRS_DEP_TIME<2100)&(bd.CRS_DEP_TIME>=1600))


In [73]:
bd.head(5)

[Row(ORIGIN_AIRPORT_ID=11433, YEAR=2024, MONTH=1, DAY_OF_MONTH=4, DAY_OF_WEEK=4, CRS_DEP_TIME=1231, TAIL_NUM='N830SK', ARR_DELAY=60.0, DEP_DELAY=72.0, ORIGIN='DTW', DEST='MBS', DISTANCE=98.0, CANCELLED=0.0, DIVERTED=0.0, CARRIER_DELAY=60.0, WEATHER_DELAY=0.0, NAS_DELAY=0.0, SECURITY_DELAY=0.0, LATE_AIRCRAFT_DELAY=0.0, Tarde=False),
 Row(ORIGIN_AIRPORT_ID=11618, YEAR=2024, MONTH=1, DAY_OF_MONTH=17, DAY_OF_WEEK=3, CRS_DEP_TIME=740, TAIL_NUM='N27263', ARR_DELAY=-12.0, DEP_DELAY=-1.0, ORIGIN='EWR', DEST='IAH', DISTANCE=1400.0, CANCELLED=0.0, DIVERTED=0.0, CARRIER_DELAY=None, WEATHER_DELAY=None, NAS_DELAY=None, SECURITY_DELAY=None, LATE_AIRCRAFT_DELAY=None, Tarde=False),
 Row(ORIGIN_AIRPORT_ID=15016, YEAR=2024, MONTH=1, DAY_OF_MONTH=14, DAY_OF_WEEK=7, CRS_DEP_TIME=1110, TAIL_NUM='N8308K', ARR_DELAY=-3.0, DEP_DELAY=5.0, ORIGIN='STL', DEST='LIT', DISTANCE=296.0, CANCELLED=0.0, DIVERTED=0.0, CARRIER_DELAY=None, WEATHER_DELAY=None, NAS_DELAY=None, SECURITY_DELAY=None, LATE_AIRCRAFT_DELAY=None, 

In [74]:
bd2 = bd.withColumn('TardeyCancelado',(bd.Tarde)&(bd.CANCELLED>0))

In [75]:
bd2.filter(bd2['TardeyCancelado'] == True)

DataFrame[ORIGIN_AIRPORT_ID: int, YEAR: int, MONTH: int, DAY_OF_MONTH: int, DAY_OF_WEEK: int, CRS_DEP_TIME: int, TAIL_NUM: string, ARR_DELAY: double, DEP_DELAY: double, ORIGIN: string, DEST: string, DISTANCE: double, CANCELLED: double, DIVERTED: double, CARRIER_DELAY: double, WEATHER_DELAY: double, NAS_DELAY: double, SECURITY_DELAY: double, LATE_AIRCRAFT_DELAY: double, Tarde: boolean, TardeyCancelado: boolean]

In [76]:
count_true = bd2.filter(bd2['TardeyCancelado'] == True)

In [77]:
bd2.count(), count_true.count()

(545759, 6440)

In [78]:
bd2 = bd2.na.fill({'CARRIER_DELAY':0, 'WEATHER_DELAY':0,'NAS_DELAY':0,'SECURITY_DELAY':0, 'LATE_AIRCRAFT_DELAY':0})

In [79]:
count_true

DataFrame[ORIGIN_AIRPORT_ID: int, YEAR: int, MONTH: int, DAY_OF_MONTH: int, DAY_OF_WEEK: int, CRS_DEP_TIME: int, TAIL_NUM: string, ARR_DELAY: double, DEP_DELAY: double, ORIGIN: string, DEST: string, DISTANCE: double, CANCELLED: double, DIVERTED: double, CARRIER_DELAY: double, WEATHER_DELAY: double, NAS_DELAY: double, SECURITY_DELAY: double, LATE_AIRCRAFT_DELAY: double, Tarde: boolean, TardeyCancelado: boolean]

In [80]:
count_tarde= bd2.filter(bd2.Tarde == True).count()
print(f'{bd.count()} vuelos fueron cancelados')

545759 vuelos fueron cancelados


In [81]:
count_true.count()*100/count_tarde

4.222203282041868

In [82]:
#count_true.explain(mode="extended")
count_true.explain(mode="formatted")
#count_true.explain(mode="cost")


== Physical Plan ==
AdaptiveSparkPlan (6)
+- Project (5)
   +- Project (4)
      +- Exchange (3)
         +- Filter (2)
            +- Scan csv  (1)


(1) Scan csv 
Output [19]: [YEAR#11316, MONTH#11317, DAY_OF_MONTH#11318, DAY_OF_WEEK#11319, TAIL_NUM#11320, ORIGIN_AIRPORT_ID#11321, ORIGIN#11324, DEST#11328, CRS_DEP_TIME#11329, DEP_DELAY#11330, ARR_DELAY#11331, CANCELLED#11332, DIVERTED#11333, DISTANCE#11334, CARRIER_DELAY#11335, WEATHER_DELAY#11336, NAS_DELAY#11337, SECURITY_DELAY#11338, LATE_AIRCRAFT_DELAY#11339]
Batched: false
Location: InMemoryFileIndex [file:/content/drive/MyDrive/BLOQUE_I.A/Datasets/T_ONTIME_REPORTING.csv]
PushedFilters: [IsNotNull(DIVERTED), IsNotNull(CRS_DEP_TIME), IsNotNull(CANCELLED), EqualTo(DIVERTED,0.0), LessThan(CRS_DEP_TIME,2100), GreaterThanOrEqual(CRS_DEP_TIME,1600), GreaterThan(CANCELLED,0.0)]
ReadSchema: struct<YEAR:int,MONTH:int,DAY_OF_MONTH:int,DAY_OF_WEEK:int,TAIL_NUM:string,ORIGIN_AIRPORT_ID:int,ORIGIN:string,DEST:string,CRS_DEP_TIME:int,DEP_DELA

In [ ]:
#TODO

#TODO

# 4 ) Dist prom canceld

In [83]:
bd2.show()

+-----------------+----+-----+------------+-----------+------------+--------+---------+---------+------+----+--------+---------+--------+-------------+-------------+---------+--------------+-------------------+-----+---------------+
|ORIGIN_AIRPORT_ID|YEAR|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|CRS_DEP_TIME|TAIL_NUM|ARR_DELAY|DEP_DELAY|ORIGIN|DEST|DISTANCE|CANCELLED|DIVERTED|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|Tarde|TardeyCancelado|
+-----------------+----+-----+------------+-----------+------------+--------+---------+---------+------+----+--------+---------+--------+-------------+-------------+---------+--------------+-------------------+-----+---------------+
|            11433|2024|    1|           4|          4|        1231|  N830SK|     60.0|     72.0|   DTW| MBS|    98.0|      0.0|     0.0|         60.0|          0.0|      0.0|           0.0|                0.0|false|          false|
|            11618|2024|    1|          17|          3|         740|

In [84]:
from pyspark.sql.functions import avg
bd2cancelled = bd2.filter(bd2.CANCELLED == True)

In [85]:
promedio_distancia = bd2cancelled.agg(avg("DISTANCE")).first()[0]

print(f"Distancia promedio de vuelos cancelados: {promedio_distancia}")

Distancia promedio de vuelos cancelados: 899.1564569130413


# 5) Airport with most  delayed percentage

In [92]:
from pyspark.sql.functions import avg, round, desc

# Retraso promedio en salidas por aeropuerto de origen
delay_por_origen = bd2.groupBy("ORIGIN").agg(avg("DEP_DELAY").alias("promedio_dep_delay")).sort(desc("promedio_dep_delay"))
delay_por_origen.show()

+------+------------------+
|ORIGIN|promedio_dep_delay|
+------+------------------+
|   ELM|             79.96|
|   CKB| 78.66666666666667|
|   STC| 67.22222222222223|
|   SMX|63.333333333333336|
|   CIU|57.870370370370374|
|   IMT|52.666666666666664|
|   PLN|            51.225|
|   PSM| 50.56521739130435|
|   CID| 47.95619047619048|
|   DEC| 47.74285714285714|
|   AVP|47.403225806451616|
|   TVC|45.708333333333336|
|   RDM| 45.57013574660633|
|   MOT| 44.76646706586826|
|   SWF|43.166666666666664|
|   FSM| 42.96341463414634|
|   CRW| 42.68852459016394|
|   MBS|42.394366197183096|
|   DLH| 40.44137931034483|
|   CSG|             38.55|
+------+------------------+
only showing top 20 rows



Después de un EDA  riguroso se busca modelar la variable Retraso2 con los siguientes features, que son los que más correlación sugieren.

# Assemble feature vector.

features y renombrar la variable respuesta
a label.
 El vector de features tendrá las columnas: DayofMonth, DayOfWeek, Tarde, Distance. La
variable de interés es Retraso2.

In [38]:
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler

from pyspark.ml.classification import LogisticRegression , RandomForestClassifier , MultilayerPerceptronClassifier


In [39]:
bd2.show()

+----+-----+------------+-----------+------------+--------+---------+---------+------+----+--------+---------+--------+-------------+-------------+---------+--------------+-------------------+-----+---------------+
|YEAR|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|CRS_DEP_TIME|TAIL_NUM|ARR_DELAY|DEP_DELAY|ORIGIN|DEST|DISTANCE|CANCELLED|DIVERTED|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|Tarde|TardeyCancelado|
+----+-----+------------+-----------+------------+--------+---------+---------+------+----+--------+---------+--------+-------------+-------------+---------+--------------+-------------------+-----+---------------+
|2024|    1|          10|          3|        1800|  N7825A|     18.0|     -3.0|   SLC| SMF|   532.0|      0.0|     0.0|          0.0|          0.0|     18.0|           0.0|                0.0| true|          false|
|2024|    1|          16|          2|        1009|  N501BG|    153.0|    154.0|   DCA| BUF|   296.0|      0.0|     0.0|          0.0|       

In [42]:

bd3 = bd2.withColumnRenamed("CANCELLED", "label")
#bd3.show()
assembler = VectorAssembler(
    inputCols=["DAY_OF_MONTH", "DAY_OF_WEEK", "Tarde", "DISTANCE"],
    outputCol="features"
)

bd_features = assembler.transform(bd3).select("features", "label")


In [43]:
bd_features.show()

+--------------------+-----+
|            features|label|
+--------------------+-----+
| [3.0,3.0,0.0,453.0]|  0.0|
|[8.0,1.0,1.0,1535.0]|  0.0|
| [2.0,2.0,0.0,547.0]|  0.0|
| [4.0,4.0,1.0,689.0]|  0.0|
| [6.0,6.0,0.0,170.0]|  0.0|
|[17.0,3.0,1.0,119...|  0.0|
|[8.0,1.0,1.0,1535.0]|  0.0|
|[12.0,5.0,0.0,296.0]|  0.0|
|[7.0,7.0,0.0,1062.0]|  0.0|
|[13.0,6.0,0.0,646.0]|  0.0|
|[1.0,1.0,0.0,1184.0]|  0.0|
|[13.0,6.0,0.0,957.0]|  1.0|
|[12.0,5.0,1.0,754.0]|  0.0|
|[15.0,1.0,0.0,308.0]|  0.0|
|[13.0,6.0,0.0,237...|  0.0|
| [1.0,1.0,0.0,507.0]|  0.0|
| [9.0,2.0,0.0,460.0]|  0.0|
|[16.0,2.0,1.0,213.0]|  0.0|
|[13.0,6.0,0.0,369.0]|  0.0|
|[4.0,4.0,0.0,4243.0]|  0.0|
+--------------------+-----+
only showing top 20 rows



In [44]:
# PARTITION THE DATA SET  (seed 42 for reproductability)
train, test = bd_features.randomSplit([0.7, 0.3], seed=42)


# 7 )Tras ejecutar el modelo de regresión logística para predecir la variable 'Cancelled' empleando las variables 'DayofMonth','DayOfWeek',"Tarde",'Distance', ¿Cuál es el coeficiente de la variable Tarde?

In [45]:
lr = LogisticRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train)

In [46]:
print("Features:", ["DAY_OF_MONTH", "DAY_OF_WEEK", "Tarde", "DISTANCE"])
print("Coeficientes:", lr_model.coefficients)
print("Intercepto:", lr_model.intercept)


Features: ['DAY_OF_MONTH', 'DAY_OF_WEEK', 'Tarde', 'DISTANCE']
Coeficientes: [-0.0025730001606938495,-0.06620854324155782,0.18379350003983289,0.00017814996683210966]
Intercepto: -3.171044813204173


In [47]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
pred = lr_model.transform(test)
evaluator = BinaryClassificationEvaluator(metricName="areaUnderROC")
auc = evaluator.evaluate(pred)
print(auc)


0.5555655220514125


In [48]:
from pyspark.ml import Pipeline


pipe = Pipeline(stages=[lr_model])

# 2) "Fit" the pipeline (no-ops for Transformers) -> PipelineModel
pipeModel = pipe.fit(train)

# 3) Use it like any model
pred_test = pipeModel.transform(test)

# 4) Evaluate
evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
print("Test AUC:", evaluator.evaluate(pred_test))

Test AUC: 0.5555618491171359


In [49]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# --- Grid: try several regularization strengths / elastic-net mixes / iterations
paramGrid = (ParamGridBuilder()
             .addGrid(lr.regParam, [0.0, 0.01, 0.1, 0.3])
             .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])  # 0=L2, 1=L1
             .addGrid(lr.maxIter, [50, 100])
             .build())

# --- Evaluator: AUC (good default for imbalanced cancellation)
evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")


In [93]:
# --- 5-fold CV (set parallelism if you have cores)
cv = CrossValidator(estimator=pipe,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=5,
                    parallelism=2)
cvModel = cv.fit(train)

In [94]:
best_lr = cvModel.bestModel
pred_test = best_lr.transform(test)

# AUC (ROC)
evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
test_auc = evaluator.evaluate(pred_test)
print("Test AUC:", test_auc)


Test AUC: 0.5555620949207969


In [95]:
bm = best_lr.stages[0]
print("Best regParam:", bm.getRegParam())
print("Best elasticNetParam:", bm.getElasticNetParam())
print("Best maxIter:", bm.getMaxIter())


Best regParam: 0.0
Best elasticNetParam: 0.0
Best maxIter: 50


In [96]:
# Accuracy (quick & visible)
test_n = pred_test.count()
test_ok = pred_test.filter(col("prediction") == col("label")).count()
print("Test Accuracy:", test_ok / test_n)

# Confusion matrix (table)
pred_test.groupBy("label","prediction").count().orderBy("label","prediction").show()




Test Accuracy: 0.9628442443446664
+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|  0.0|       0.0|157400|
|  1.0|       0.0|  6074|
+-----+----------+------+



Nice—those params make sense for a first pass:

Best regParam = 0.10
Strength of regularization (λ). Higher → stronger shrinkage of all weights toward 0. Helps fight overfitting & multicollinearity, but can underfit if too large.

Best elasticNetParam = 0.0
Mix between L2 (ridge) and L1 (lasso).

0.0 = pure L2: smooth shrinkage, keeps most features non-zero.

1.0 = pure L1: drives some weights exactly to 0 (feature selection).
Your CV preferred L2—common when all features carry some signal and you don’t need sparsity.

Best maxIter = 100
Optimization budget for LBFGS (default). If you see convergence warnings, increase; otherwise it just sets an upper bound.

Given your AUC ≈ 0.654, the model is better than random but modest—reasonable with only 4 simple features (date, weekday, late flag, distance) for predicting cancellation, which is rare and driven by factors you didn’t include (weather, carrier ops, ATC).

DayofMonth (coef = +0.055, OR ≈ 1.057)

Each extra day later in the month increases the odds of a cancellation by ~5.7% (holding other variables constant).

Not huge, but suggests some month-end patterns (maybe scheduling or reporting quirks).

DayOfWeek (coef = –0.014, OR ≈ 0.986)

Each increment in weekday number slightly reduces odds (~1.4% per day).

Very small, so effect isn’t strong — cancellations are fairly uniform across weekdays in this slice.

Tarde (coef = +0.198, OR ≈ 1.219)

If a flight is flagged late (Tarde=1), the odds of being cancelled rise by ~22%.

This is the most interpretable and impactful feature here. You can point at it in class: late flights are more likely to be cancelled.

Distance (coef ≈ –0.00033, OR ≈ 0.9997)

For every extra mile, odds of cancellation decrease by ~0.03%.

Negligible at small scales, but across 1000 miles you’d see ~26% reduction.

Interpretable as: longer-haul flights are slightly less likely to be cancelled than short hops.


# 7)Tras ejecutar un bosque aleatorio en el contexto anterior, considerando las mismas variables, ¿cuánto vale el área bajo la curva evaluada en la base de datos test?

In [98]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=100)
model = rf.fit(train)
pred = model.transform(test)

In [99]:
pred = model.transform(bd2_test)
evaluator = BinaryClassificationEvaluator(metricName="areaUnderROC")
auc = evaluator.evaluate(pred)
print(auc)


NameError: name 'bd2_test' is not defined

# 8 ) Red Neuronal

In [ ]:
mlp= MultilayerPerceptronClassifier(labelCol="label",
featuresCol="features", maxIter=100,
layers=[4, 5, 2], seed=123)

In [ ]:
model = mlp.fit(bd2_train)

pred = model.transform(bd2_test)

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator as MCCE

evaluator = MCCE(metricName="precision")
evaluator.evaluate(pred)


# 11 ) FIlter cancelled and Delayed and analyze the following

In [ ]:
bd = bd.filter((bd.Cancelled == 0)&(bd.Diverted == 0))

In [ ]:
bd = bd.withColumn('Retraso2',(bd.ArrDelay - bd.LateAircraftDelay))

In [ ]:
bd = bd.withColumn('Tarde',(bd.CRSDepTime<2100)&(bd.CRSDepTime>=1600))

In [ ]:
bd.columns